In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
from typing import Optional
from pathlib import Path
import json
import shutil

In [ ]:
def make_input_folder(
    model_name_and_path: Optional[tuple[str, Path]],
    inputs: list[tuple[Path, str]],
    dest: Path,
):
    dest.mkdir(exist_ok=True, parents=True)
    meta = {}

    # save model first
    if model_name_and_path is not None:
        name, path = model_name_and_path
        shutil.copy(path, dest / path.name)
        meta["model"] = {
            "name": name,
            "path": path.name,
        }

    meta["inputs"] = []
    inputs_dir = dest / "inputs"
    inputs_dir.mkdir(parents=True, exist_ok=True)
    for input_path, label in inputs:
        dest_input_path = inputs_dir / input_path.name
        shutil.copy(input_path, dest_input_path)
        meta["inputs"].append(
            {
                "path": str(Path("inputs") / input_path.name),
                "label": label,
            }
        )
    
    with open(dest / "meta.json", "w") as f:
        json.dump(meta, f)

In [ ]:
from pt_to_api.mnist import get_learner, get_mnist_dataloader
learner = get_learner()

In [ ]:
dl = get_mnist_dataloader(0.0005, bs=4)

In [ ]:
len(dl.train.items)

In [ ]:
import random
import string

def get_random_string(length):
    # Choose from letters (a-z, A-Z) and digits (0-9)
    characters = string.ascii_letters + string.digits
    return ''.join(random.choices(characters, k=length))

In [ ]:
import torch
import tempfile

ORIG_MODEL_PT_FILE = Path("/Users/hariomnarang/Desktop/personal/hiccup-ide/backend/test_load/model.pt")
model_name = "simple_mnist_v1"
dloader = dl.train
DEST = Path("../folder_with_all_inputs")
DEST.mkdir(exist_ok=True, parents=True)

with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)

    model_pt = tmp / "model.pt"
    shutil.copy(ORIG_MODEL_PT_FILE, model_pt)
    model_name_and_path = (model_name, model_pt)

    inputs_and_labels = []
    for batch, labels in dloader:
        for i in range(len(batch)):
            inp_tens, label = batch[i], labels[i].item()
            inp_tens = inp_tens.unsqueeze(0)
            fname = f"{label}-{get_random_string(3)}.pt"
            dest = tmp / fname
            torch.save(inp_tens, dest)
            inputs_and_labels.append((dest, label))
    make_input_folder(model_name_and_path, inputs_and_labels, DEST)

In [ ]:
inputs_and_labels[0]

In [ ]:
for batch, labels in dl.train:
    print(batch.shape, labels.shape)